# Gowalla TDPPG Sensitivity Analysis Only

This notebook builds a Temporal Detailed People Proximity Graph, or TDPPG, for the **Gowalla** check-in dataset.

The goal is to detect repeated user-user proximity patterns from location-based social network check-ins. Users are treated as graph nodes, and repeated spatiotemporal encounters are converted into weighted edges. The final graph is filtered, partitioned using PyMetis, and evaluated using the social graph as an indirect validation signal.

This is the backup sensitivity-only notebook for **Gowalla**.

It keeps the same TDPPG helper functions as the main notebook, but the main purpose is to rerun the sensitivity tests for distance threshold, time gap, and maximum event group size.

## Method summary

The pipeline follows these steps:

1. Load check-ins and social edges.
2. Select the most active users to keep the experiment manageable.
3. Convert check-ins into venue/time/spatial events.
4. Create valid user-pair encounters using distance and time constraints.
5. Aggregate pair-level features:
   - encounter count
   - encounter strength
   - distinct days
   - shared venues
6. Build the weighted TDPPG graph.
7. Filter weak edges.
8. Partition the graph with PyMetis.
9. Evaluate ranking and partition quality.

In [ ]:
# Google Colab setup
# Run this cell if pymetis is not already installed.
try:
    import pymetis
    print("pymetis already installed")
except ImportError:
    import sys
    !{sys.executable} -m pip -q install pymetis networkx scikit-learn pyarrow

In [ ]:
import os
import gzip
import json
import math
import time
import itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import pymetis

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

DATASET = "gowalla"
OUTPUT_DIR = Path("outputs") / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Keep this smaller if running on Colab free tier.
TOP_N_USERS = 3000

# Final parameters from the report.
MAX_DISTANCE_M = 150
MAX_TIME_GAP_SECONDS = 3600
MIN_EVENT_SIZE = 2
MAX_EVENT_GROUP_SIZE = 25
N_PARTITIONS = 20

print("Dataset:", DATASET)
print("Output directory:", OUTPUT_DIR)

## Load data

Expected SNAP files:

- `loc-gowalla_totalCheckins.txt.gz`
- `loc-gowalla_edges.txt.gz`
- `loc-brightkite_totalCheckins.txt.gz`
- `loc-brightkite_edges.txt.gz`

Put them in `data/`, `/content/`, or `/mnt/data/`. The notebook checks all common paths.

In [ ]:
def find_file(filename):
    search_dirs = [
        Path("data"),
        Path("../data"),
        Path("/content"),
        Path("/content/data"),
        Path("/mnt/data"),
        Path("/mnt/data/data"),
    ]

    for directory in search_dirs:
        candidate = directory / filename
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not find {filename}. Put it in data/, /content/, or /mnt/data/."
    )


def dataset_files(dataset):
    if dataset == "gowalla":
        return {
            "checkins": "loc-gowalla_totalCheckins.txt.gz",
            "edges": "loc-gowalla_edges.txt.gz",
        }
    if dataset == "brightkite":
        return {
            "checkins": "loc-brightkite_totalCheckins.txt.gz",
            "edges": "loc-brightkite_edges.txt.gz",
        }
    raise ValueError("dataset must be 'gowalla' or 'brightkite'")


files = dataset_files(DATASET)
checkin_path = find_file(files["checkins"])
edge_path = find_file(files["edges"])

print("Check-ins:", checkin_path)
print("Social edges:", edge_path)

In [ ]:
def load_checkins(path):
    cols = ["user_id", "checkin_time", "latitude", "longitude", "venue_id"]
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=cols,
        compression="infer",
        parse_dates=["checkin_time"],
    )

    df["user_id"] = df["user_id"].astype(str)
    df["venue_id"] = df["venue_id"].astype(str)
    df = df.dropna(subset=["checkin_time", "latitude", "longitude", "venue_id"])
    df = df.drop_duplicates()
    return df


def load_social_edges(path):
    edges = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["user_a", "user_b"],
        compression="infer",
    )

    edges["user_a"] = edges["user_a"].astype(str)
    edges["user_b"] = edges["user_b"].astype(str)

    edges["u"] = edges[["user_a", "user_b"]].min(axis=1)
    edges["v"] = edges[["user_a", "user_b"]].max(axis=1)
    edges = edges[["u", "v"]].drop_duplicates()
    return edges


checkins_raw = load_checkins(checkin_path)
social_edges_raw = load_social_edges(edge_path)

print("Raw check-ins:", checkins_raw.shape)
print("Raw social edges:", social_edges_raw.shape)
checkins_raw.head()

## Select active users

The full SNAP datasets can be large. For a class project experiment, we keep the most active users so that pair-event construction remains manageable.

In [ ]:
def filter_active_users(checkins, social_edges, top_n_users=3000):
    active_users = checkins["user_id"].value_counts().head(top_n_users).index.astype(str)

    checkins_small = checkins[checkins["user_id"].isin(active_users)].copy()
    social_edges_small = social_edges[
        social_edges["u"].isin(active_users) & social_edges["v"].isin(active_users)
    ].copy()

    return checkins_small, social_edges_small, set(active_users)


checkins, social_edges, active_user_set = filter_active_users(
    checkins_raw,
    social_edges_raw,
    top_n_users=TOP_N_USERS,
)

print("Filtered check-ins:", checkins.shape)
print("Filtered social edges:", social_edges.shape)
print("Unique users:", checkins["user_id"].nunique())
print("Unique venues:", checkins["venue_id"].nunique())

## TDPPG helper functions

The encounter score gives more credit to check-ins that are closer in both space and time.

The final edge weight combines:

\[
W_{ij} \propto C_{ij} + S_{ij} + D_{ij} + V_{ij}
\]

where:

- \(C_{ij}\) = encounter count
- \(S_{ij}\) = encounter strength
- \(D_{ij}\) = number of distinct encounter days
- \(V_{ij}\) = number of shared venues

In [ ]:
EARTH_RADIUS_M = 6_371_000


def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_M * c


def pair_key(u, v):
    return tuple(sorted((str(u), str(v))))


def add_event_features(df, max_distance_m, max_time_gap_seconds):
    out = df.copy()

    out["checkin_time"] = pd.to_datetime(out["checkin_time"], utc=True, errors="coerce")
    out = out.dropna(subset=["checkin_time", "latitude", "longitude"])

    out["event_day"] = out["checkin_time"].dt.date.astype(str)
    out["time_bin"] = (
        out["checkin_time"].astype("int64") // (max_time_gap_seconds * 1_000_000_000)
    )

    # Rough spatial cell. This keeps candidate groups small.
    # 1 degree latitude is about 111.32 km.
    cell_size = max_distance_m / 111_320
    out["lat_cell"] = np.floor(out["latitude"] / cell_size).astype(int)
    out["lon_cell"] = np.floor(out["longitude"] / cell_size).astype(int)

    return out


def build_pair_events(
    checkins,
    max_distance_m=150,
    max_time_gap_seconds=3600,
    min_event_size=2,
    max_event_group_size=25,
):
    df = add_event_features(checkins, max_distance_m, max_time_gap_seconds)

    pair_rows = []
    grouped = df.groupby(["time_bin", "lat_cell", "lon_cell"], sort=False)

    for _, group in grouped:
        group = group.drop_duplicates(subset=["user_id", "venue_id", "checkin_time"])

        n_users = group["user_id"].nunique()
        if n_users < min_event_size or n_users > max_event_group_size:
            continue

        records = group[
            ["user_id", "venue_id", "checkin_time", "event_day", "latitude", "longitude"]
        ].to_dict("records")

        for a, b in itertools.combinations(records, 2):
            if a["user_id"] == b["user_id"]:
                continue

            time_diff = abs((a["checkin_time"] - b["checkin_time"]).total_seconds())
            if time_diff > max_time_gap_seconds:
                continue

            distance = haversine_m(
                a["latitude"],
                a["longitude"],
                b["latitude"],
                b["longitude"],
            )

            if distance > max_distance_m:
                continue

            u, v = pair_key(a["user_id"], b["user_id"])

            # Smooth strength score: closer in space and time gets higher weight.
            spatial_score = 1 - (distance / max_distance_m)
            temporal_score = 1 - (time_diff / max_time_gap_seconds)
            encounter_strength = max(0, spatial_score) * max(0, temporal_score)

            pair_rows.append(
                {
                    "u": u,
                    "v": v,
                    "event_day": a["event_day"],
                    "venue_a": a["venue_id"],
                    "venue_b": b["venue_id"],
                    "distance_m": float(distance),
                    "time_diff_seconds": float(time_diff),
                    "encounter_strength": float(encounter_strength),
                }
            )

    return pd.DataFrame(pair_rows)


def aggregate_tdppg_pairs(pair_events):
    if pair_events.empty:
        return pd.DataFrame(
            columns=[
                "u",
                "v",
                "encounter_count",
                "encounter_strength",
                "distinct_days",
                "shared_venues",
                "avg_distance_m",
                "avg_time_diff_seconds",
                "edge_weight",
            ]
        )

    events = pair_events.copy()
    events["same_venue"] = events["venue_a"] == events["venue_b"]

    agg = (
        events.groupby(["u", "v"])
        .agg(
            encounter_count=("event_day", "size"),
            encounter_strength=("encounter_strength", "sum"),
            distinct_days=("event_day", "nunique"),
            shared_venues=("same_venue", "sum"),
            avg_distance_m=("distance_m", "mean"),
            avg_time_diff_seconds=("time_diff_seconds", "mean"),
        )
        .reset_index()
    )

    # Normalize components before combining them so one feature does not dominate.
    for col in ["encounter_count", "encounter_strength", "distinct_days", "shared_venues"]:
        max_val = agg[col].max()
        if max_val and max_val > 0:
            agg[f"{col}_norm"] = agg[col] / max_val
        else:
            agg[f"{col}_norm"] = 0

    agg["edge_weight"] = (
        agg["encounter_count_norm"]
        + agg["encounter_strength_norm"]
        + agg["distinct_days_norm"]
        + agg["shared_venues_norm"]
    )

    return agg.sort_values("edge_weight", ascending=False).reset_index(drop=True)

## Build TDPPG pair events and weighted edges

In [ ]:
start = time.perf_counter()

pair_events = build_pair_events(
    checkins,
    max_distance_m=MAX_DISTANCE_M,
    max_time_gap_seconds=MAX_TIME_GAP_SECONDS,
    min_event_size=MIN_EVENT_SIZE,
    max_event_group_size=MAX_EVENT_GROUP_SIZE,
)

pair_event_runtime = time.perf_counter() - start

print("Pair events:", pair_events.shape)
print("Pair-event construction runtime:", round(pair_event_runtime, 2), "seconds")

pair_events.head()

In [ ]:
tdppg_pairs = aggregate_tdppg_pairs(pair_events)

print("TDPPG pairs:", tdppg_pairs.shape)
tdppg_pairs.head()

## Build graph and partition with PyMetis

In [ ]:
def build_graph(tdppg_pairs, min_edge_weight=None):
    pairs = tdppg_pairs.copy()

    if min_edge_weight is None:
        if len(pairs) == 0:
            min_edge_weight = 0
        else:
            min_edge_weight = pairs["edge_weight"].quantile(0.25)

    pairs = pairs[pairs["edge_weight"] >= min_edge_weight].copy()

    graph = nx.Graph()
    for row in pairs.itertuples(index=False):
        graph.add_edge(row.u, row.v, weight=float(row.edge_weight))

    return graph, pairs, min_edge_weight


def partition_graph(graph, n_partitions=20):
    nodes = list(graph.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    if len(nodes) == 0:
        return pd.DataFrame(columns=["user_id", "partition"])

    adjacency = []
    for node in nodes:
        neighbors = [node_to_idx[nbr] for nbr in graph.neighbors(node)]
        adjacency.append(neighbors)

    actual_partitions = min(n_partitions, max(1, len(nodes)))

    _, memberships = pymetis.part_graph(actual_partitions, adjacency=adjacency)

    return pd.DataFrame(
        {
            "user_id": nodes,
            "partition": memberships,
        }
    )


graph_start = time.perf_counter()

tdppg_graph, filtered_pairs, min_edge_weight = build_graph(tdppg_pairs)
partitions = partition_graph(tdppg_graph, n_partitions=N_PARTITIONS)

partition_runtime = time.perf_counter() - graph_start

print("Min edge weight:", round(min_edge_weight, 4))
print("Graph nodes:", tdppg_graph.number_of_nodes())
print("Graph edges:", tdppg_graph.number_of_edges())
print("Partition runtime:", round(partition_runtime, 4), "seconds")

partitions.head()

## Evaluate TDPPG ranking and partition quality

Social links are not direct co-travel labels, but they are useful as an indirect validation signal. If the TDPPG ranking is meaningful, known social edges should appear more often among the high-scoring user pairs.

In [ ]:
def make_social_edge_set(social_edges):
    return set(map(tuple, social_edges[["u", "v"]].values))


def evaluate_pair_ranking(tdppg_pairs, social_edges, k_values=(100, 500, 1000)):
    if tdppg_pairs.empty:
        return {
            "roc_auc": np.nan,
            "pr_auc": np.nan,
            **{f"p_at_{k}": np.nan for k in k_values},
        }

    social_set = make_social_edge_set(social_edges)

    eval_df = tdppg_pairs[["u", "v", "edge_weight"]].copy()
    eval_df["label"] = eval_df.apply(lambda r: (r["u"], r["v"]) in social_set, axis=1)

    # If all labels are the same, AUC is undefined.
    if eval_df["label"].nunique() < 2:
        roc_auc = np.nan
        pr_auc = np.nan
    else:
        roc_auc = roc_auc_score(eval_df["label"], eval_df["edge_weight"])
        pr_auc = average_precision_score(eval_df["label"], eval_df["edge_weight"])

    out = {
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

    ranked = eval_df.sort_values("edge_weight", ascending=False)
    for k in k_values:
        top_k = ranked.head(k)
        out[f"p_at_{k}"] = top_k["label"].mean() if len(top_k) else np.nan

    return out


def same_partition_social_edge_rate(partitions, social_edges):
    if partitions.empty or social_edges.empty:
        return np.nan

    part_map = dict(zip(partitions["user_id"], partitions["partition"]))

    valid = 0
    same = 0

    for row in social_edges.itertuples(index=False):
        if row.u in part_map and row.v in part_map:
            valid += 1
            same += int(part_map[row.u] == part_map[row.v])

    return same / valid if valid else np.nan


metrics = evaluate_pair_ranking(tdppg_pairs, social_edges)
metrics["same_partition_social_edge_rate"] = same_partition_social_edge_rate(partitions, social_edges)
metrics["pair_event_runtime_seconds"] = pair_event_runtime
metrics["partition_runtime_seconds"] = partition_runtime
metrics["graph_nodes"] = tdppg_graph.number_of_nodes()
metrics["graph_edges"] = tdppg_graph.number_of_edges()

metrics_df = pd.DataFrame([metrics])
metrics_df

## Save outputs

In [ ]:
pair_events.to_json(OUTPUT_DIR / "pair_events.jsonl", orient="records", lines=True)
tdppg_pairs.to_csv(OUTPUT_DIR / "tdppg_pairs.csv", index=False)
partitions.to_csv(OUTPUT_DIR / "tdppg_partitions.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "pair_eval_table.csv", index=False)

runtime_summary = pd.DataFrame(
    [
        {
            "dataset": DATASET,
            "pair_event_runtime_seconds": pair_event_runtime,
            "partition_runtime_seconds": partition_runtime,
            "graph_nodes": tdppg_graph.number_of_nodes(),
            "graph_edges": tdppg_graph.number_of_edges(),
        }
    ]
)

runtime_summary.to_csv(OUTPUT_DIR / "runtime_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)

## Quick visual checks

In [ ]:
import matplotlib.pyplot as plt

if not tdppg_pairs.empty:
    plt.figure(figsize=(8, 5))
    tdppg_pairs["edge_weight"].hist(bins=50)
    plt.title(f"{DATASET.capitalize()} TDPPG edge-weight distribution")
    plt.xlabel("Edge weight")
    plt.ylabel("Number of user pairs")
    plt.show()

    plt.figure(figsize=(8, 5))
    tdppg_pairs.head(100)["edge_weight"].reset_index(drop=True).plot()
    plt.title(f"{DATASET.capitalize()} top-100 TDPPG edge weights")
    plt.xlabel("Rank")
    plt.ylabel("Edge weight")
    plt.show()
else:
    print("No TDPPG pairs found. Try increasing TOP_N_USERS or relaxing thresholds.")

## Sensitivity analysis

This section tests how the graph quality changes when we modify the main TDPPG parameters.

In [ ]:
def run_tdppg_experiment(
    checkins,
    social_edges,
    dataset,
    max_distance_m=150,
    max_time_gap_seconds=3600,
    max_event_group_size=25,
    n_partitions=20,
    label="experiment",
):
    start = time.perf_counter()

    events = build_pair_events(
        checkins,
        max_distance_m=max_distance_m,
        max_time_gap_seconds=max_time_gap_seconds,
        min_event_size=MIN_EVENT_SIZE,
        max_event_group_size=max_event_group_size,
    )

    pair_runtime = time.perf_counter() - start

    pairs = aggregate_tdppg_pairs(events)

    graph_start = time.perf_counter()
    graph, filtered, min_weight = build_graph(pairs)
    parts = partition_graph(graph, n_partitions=n_partitions)
    part_runtime = time.perf_counter() - graph_start

    result = evaluate_pair_ranking(pairs, social_edges)
    result.update(
        {
            "dataset": dataset,
            "label": label,
            "max_distance_m": max_distance_m,
            "max_time_gap_seconds": max_time_gap_seconds,
            "max_event_group_size": max_event_group_size,
            "pair_events": len(events),
            "tdppg_pairs": len(pairs),
            "graph_nodes": graph.number_of_nodes(),
            "graph_edges": graph.number_of_edges(),
            "same_partition_social_edge_rate": same_partition_social_edge_rate(parts, social_edges),
            "pair_event_runtime_seconds": pair_runtime,
            "partition_runtime_seconds": part_runtime,
        }
    )

    return result


sensitivity_runs = []

for distance in [75, 150, 300]:
    sensitivity_runs.append(
        run_tdppg_experiment(
            checkins,
            social_edges,
            DATASET,
            max_distance_m=distance,
            max_time_gap_seconds=MAX_TIME_GAP_SECONDS,
            max_event_group_size=MAX_EVENT_GROUP_SIZE,
            label=f"distance_{distance}m",
        )
    )

for gap in [1800, 3600, 7200]:
    sensitivity_runs.append(
        run_tdppg_experiment(
            checkins,
            social_edges,
            DATASET,
            max_distance_m=MAX_DISTANCE_M,
            max_time_gap_seconds=gap,
            max_event_group_size=MAX_EVENT_GROUP_SIZE,
            label=f"time_gap_{gap}s",
        )
    )

for group_size in [10, 25, 50]:
    sensitivity_runs.append(
        run_tdppg_experiment(
            checkins,
            social_edges,
            DATASET,
            max_distance_m=MAX_DISTANCE_M,
            max_time_gap_seconds=MAX_TIME_GAP_SECONDS,
            max_event_group_size=group_size,
            label=f"group_size_{group_size}",
        )
    )

sensitivity_df = pd.DataFrame(sensitivity_runs)
sensitivity_df

In [ ]:
sensitivity_df.to_csv(OUTPUT_DIR / "sensitivity_summary.csv", index=False)
print("Saved sensitivity summary to:", OUTPUT_DIR / "sensitivity_summary.csv")

In [ ]:
import matplotlib.pyplot as plt

metric = "pr_auc"

if metric in sensitivity_df.columns:
    plot_df = sensitivity_df[["label", metric]].dropna()

    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["label"], plot_df[metric])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{DATASET.capitalize()} sensitivity analysis: {metric}")
    plt.xlabel("Run")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()